# Test `resolution_compt.py`

This notebook calls an external 2pac executable through `resolution_compt.py`.

- 2pac computes a minimal free resolution of the selected two-parameter persistent homology module over the ambient `\mathbb{Z}^2` grid.
- The finite-grid boundary-cone workflow uses 2pac as an unmodified executable and computes finite-grid minimal projective resolutions and minimal injective coresolutions.

The external 2pac program is not bundled in this repository. Install it separately and then set the input path below.

In [ ]:
import json
from pathlib import Path

from resolution_compt import (
    InputKind,
    compute_finite_grid_injective_coresolution,
    is_executable_available,
    is_twopac_python_available,
    validate_scc2020_file,
)
from finite_grid_cone import (
    read_scc2020_chain_complex,
    compute_grid_compression,
)

PROJECT_ROOT = Path.cwd()
print(PROJECT_ROOT)

## Configuration

Set `INPUT_PATH` to the bifiltered input file you want to test. This notebook is configured for a standard complete `scc2020` file with two persistence parameters.

Typical choice:

- `InputKind.SCC2020`: the finite-grid workflow validates and reads a complete `scc2020` chain complex.

Other 2pac-supported input formats can be adapted separately, but this notebook intentionally keeps only the `scc2020` finite-grid workflow.

In [ ]:
# Change this path before running the computation cells.
INPUT_PATH = PROJECT_ROOT / "data" / "chain_complex_scc2020" / "3x3_bifiltration.txt"
# INPUT_PATH = PROJECT_ROOT / "data" / "chain_complex_scc2020" / "two_circles_200pts_knn_function.alpha.scc2020.txt"
# INPUT_PATH = PROJECT_ROOT / "data" / "chain_complex_scc2020" / "chromatic_four_circles_500pts_function.alpha.scc2020.txt"
if not INPUT_PATH.exists():
    txt_candidate = Path(str(INPUT_PATH) + ".txt")
    if txt_candidate.exists():
        INPUT_PATH = txt_candidate
INPUT_KIND = InputKind.SCC2020

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "resolution_computation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Use the executable name if it is on PATH, or an absolute path otherwise.
TWOPAC_BINARY = "2pac"

# 2pac's --dim parameter computes homology dimensions 0..TWOPAC_MAXDIM.
# With --save-resolution-scc, 2pac writes only the resolution for H_TWOPAC_MAXDIM.
# Use 1 to save H_1; use 2 to save H_2.
TWOPAC_MAXDIM = 1

# Input grade metadata for the selected complete scc2020 chain complex.
INPUT_CHAIN_COMPLEX = read_scc2020_chain_complex(INPUT_PATH)
INPUT_GRID_COMPRESSION = compute_grid_compression(INPUT_CHAIN_COMPLEX)


def infer_input_grade_min_max(chain_complex):
    grades = [grade for term_grades in chain_complex.term_grades for grade in term_grades]
    if not grades:
        raise ValueError("Cannot infer grade bounds from an empty scc2020 complex.")
    num_parameters = len(grades[0])
    grid_min = tuple(min(grade[index] for grade in grades) for index in range(num_parameters))
    grid_max = tuple(max(grade[index] for grade in grades) for index in range(num_parameters))
    return grid_min, grid_max


INPUT_GRADE_MIN, INPUT_GRADE_MAX = infer_input_grade_min_max(INPUT_CHAIN_COMPLEX)
TIMEOUT_SECONDS = None

print("Input:", INPUT_PATH)
print("Input exists:", INPUT_PATH.exists())
print("Output directory:", OUTPUT_DIR)
print("Input grade min:", INPUT_GRADE_MIN)
print("Input grade max:", INPUT_GRADE_MAX)
print("Compressed finite-grid upper:", INPUT_GRID_COMPRESSION.upper)

## Environment Check


In [ ]:
tool_status = {
    "2pac_executable": is_executable_available(TWOPAC_BINARY),
    "twopac_python_binding": is_twopac_python_available(),
}
tool_status

In [ ]:
def require_input_file(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(
            f"Set INPUT_PATH to an existing input file before running computations: {path}"
        )
    if not path.is_file():
        raise ValueError(f"INPUT_PATH must be a file: {path}")


def preview_text_file(path: Path, max_lines: int = 30) -> None:
    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    for line in lines[:max_lines]:
        print(line)
    if len(lines) > max_lines:
        print(f"... ({len(lines) - max_lines} more lines)")


def print_result(result) -> None:
    print("Backend:", result.backend.value)
    print("Resolution kind:", result.resolution_kind.value)
    print("Output:", result.output_path)
    print("Elapsed seconds:", round(result.elapsed_seconds, 3))
    print("Command:", " ".join(result.command))
    if result.stdout:
        print("\nSTDOUT:\n", result.stdout)
    if result.stderr:
        print("\nSTDERR:\n", result.stderr)



def format_grade(grade):
    return "(" + ", ".join(str(coordinate) for coordinate in grade) + ")"


def format_grade_list(grades):
    return [format_grade(grade) for grade in grades]


## scc2020 Standard Check

The Lesnick-Kerber scc2020 proposal ignores empty lines and lines starting with `#`. The first meaningful lines should be:

1. `scc2020`
2. number of persistence parameters `d`
3. generator sizes `m_1 ... m_{n+1}`
4. optional `--reverse ...` and `--field ...` lines

For this notebook, `d` should be `2` because we are testing bifiltrations.


In [ ]:
require_input_file(INPUT_PATH)

if INPUT_KIND != InputKind.SCC2020:
    raise RuntimeError(
        "This notebook expects a standard complete scc2020 input. "
        "Convert other formats to scc2020 first, or adapt the calls below."
    )

scc_validation = validate_scc2020_file(INPUT_PATH)
scc_summary = scc_validation.summary

print("valid:", scc_validation.is_valid)
print("num_parameters:", scc_summary.num_parameters)
print("generator_sizes:", scc_summary.generator_sizes)
print("reverse_coordinates:", scc_summary.reverse_coordinates)
print("field:", scc_summary.field)
print("data_start_line_number:", scc_summary.data_start_line_number)
print("actual_data_lines:", scc_validation.actual_data_lines)
print("expected_without_final_block:", scc_validation.expected_data_lines_without_final_block)
print("expected_with_final_block:", scc_validation.expected_data_lines_with_final_block)
print("has_final_grade_block:", scc_validation.has_final_grade_block)

if scc_validation.warnings:
    print("\nWarnings:")
    for warning in scc_validation.warnings:
        print("-", warning)

if scc_validation.errors:
    print("\nErrors:")
    for error in scc_validation.errors:
        print("-", error)
    raise RuntimeError("INPUT_PATH is not a valid scc2020 file.")

if scc_summary.num_parameters != 2:
    raise RuntimeError(
        f"This notebook is for 2-parameter input, but d={scc_summary.num_parameters}."
    )

## Finite-Grid Persistent Homology Workflow

This workflow computes minimal projective resolutions and minimal injective coresolutions associated with a persistence module indexed by a finite 2D grid.

It builds a finite-boundary cone and computes the zero extension outside the chosen grid. 2pac is used as an unmodified external executable inside this workflow.

## Finite-Boundary Cone / Zero Extension

In [ ]:
import inspect

if "save_json" not in inspect.signature(compute_finite_grid_injective_coresolution).parameters:
    raise RuntimeError(
        "resolution_compt.py has been updated after this notebook imported it. "
        "Restart the kernel, then rerun the import and configuration cells before this workflow."
    )

finite_grid_cone_result = compute_finite_grid_injective_coresolution(
    INPUT_PATH,
    input_kind=INPUT_KIND,
    output_dir=OUTPUT_DIR,
    output_prefix=INPUT_PATH.stem,
    twopac_binary=TWOPAC_BINARY,
    target_dim=TWOPAC_MAXDIM,
    timeout=TIMEOUT_SECONDS,
    save_json=False,
)

finite_projective_output = finite_grid_cone_result.projective_txt_path
finite_injective_output = finite_grid_cone_result.injective_txt_path


def format_grade_list(grades):
    return [format_grade(grade) for grade in grades]


print("Compressed grid metadata:")
print(json.dumps(finite_grid_cone_result.compression.to_dict(), indent=2))

print("\nFinite-grid projective resolution, minimal expected:")
for degree, grades in enumerate(finite_grid_cone_result.projective_resolution.term_grades):
    print(f"P_{degree}:", format_grade_list(grades))

print("\nFinite-grid injective coresolution with Q_G summands, minimal expected:")
for q, grades in enumerate(finite_grid_cone_result.injective_coresolution.terms):
    print(f"Q^{q}:", format_grade_list(grades))

print("\nGenerated outputs:")
print("temporary coned complex txt:", finite_grid_cone_result.coned_complex_path)
print("zero-extended free resolution txt:", finite_grid_cone_result.zero_extended_resolution_path or "deleted after extraction")
print("finite-grid projective txt:", finite_projective_output)
print("finite-grid injective txt:", finite_injective_output)

print("\nFinite-grid projective txt preview:")
preview_text_file(finite_projective_output, max_lines=40)
print("\nFinite-grid injective txt preview:")
preview_text_file(finite_injective_output, max_lines=40)


## Interpreting the Finite-Grid Output

The finite-boundary workflow builds a zero extension from the input chain complex, compresses all finite input grades, and then computes the corresponding finite-grid projective and injective outputs.

For each input file, the actual finite grid is determined from the data. The code cell above prints the compressed grid metadata and writes the current projective and injective outputs to txt files.

The projective summands are written as `P_G(...)`. The injective summands are written as `Q_G(...)`. The displayed grades are compressed grid coordinates; the printed JSON metadata records the original coordinate values used for the compression.

## Downstream Variables

The finite-grid boundary-cone workflow writes scc2020-style `.txt` files for downstream computations and easy inspection.

In [ ]:
computed_outputs = {
    "finite_grid_coned_complex_txt": finite_grid_cone_result.coned_complex_path,
    "finite_grid_projective_resolution_txt": finite_projective_output,
    "finite_grid_injective_coresolution_txt": finite_injective_output,
}

computed_outputs